# Qwen3-VL-4B-Instruct Inference trên MMHal-Bench (Kaggle T4 GPU)

Notebook này được thiết kế để chạy trực tiếp trên **Kaggle GPU (T4 x2)** nhằm thu thập kết quả suy luận thực tế của mô hình `Qwen/Qwen3-VL-4B-Instruct` trên 96 mẫu của dataset `Shengcao1006/MMHal-Bench`.

### Yêu cầu cài đặt trên Kaggle:
1. **Accelerator:** Chọn `GPU T4 x2` hoặc `GPU P100`.
2. **Internet:** Gạt sang `ON`.

## 1. Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q --upgrade pip
!pip install -q transformers>=4.48.0 accelerate>=1.0.0 torchvision pillow qwen-vl-utils pyyaml

## 2. Kiểm tra GPU & VRAM

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    raise SystemError("Vui lòng bật GPU trong Notebook Settings (Accelerator -> GPU T4 x2)!")

## 3. Tải dataset MMHal-Bench và nạp mô hình Qwen3-VL-4B-Instruct

In [ ]:
import os
import json
import time
import zipfile
import urllib.request
from pathlib import Path
from datetime import datetime, timezone
from PIL import Image
from transformers import AutoProcessor

# 1. Tải và giải nén test_data.zip trực tiếp (Tránh lỗi dataset script bị cấm trên datasets 4.0)
DATA_DIR = Path("/kaggle/working/mmhal_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
zip_url = "https://huggingface.co/datasets/Shengcao1006/MMHal-Bench/resolve/main/test_data.zip"
zip_path = DATA_DIR / "test_data.zip"

if not (DATA_DIR / "response_template.json").exists():
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Đang tải dữ liệu MMHal-Bench (test_data.zip, ~170MB)...")
    urllib.request.urlretrieve(zip_url, zip_path)
    print("-> Đang giải nén hình ảnh và câu hỏi...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATA_DIR)
    print("-> Đã giải nén xong!")

# 2. Đọc 96 mẫu từ response_template.json
template_file = DATA_DIR / "response_template.json"
with open(template_file, "r", encoding="utf-8") as f:
    dataset = json.load(f)

total_samples = len(dataset)
print(f"-> Tổng số mẫu MMHal-Bench nạp thành công: {total_samples}")

# 3. Tự động nhận diện class tương thích linh hoạt cho Qwen3-VL
try:
    from transformers import Qwen3VLForConditionalGeneration as VLMClass
except Exception:
    try:
        from transformers import AutoModelForImageTextToText as VLMClass
    except Exception:
        try:
            from transformers import Qwen2_5_VLForConditionalGeneration as VLMClass
        except Exception:
            from transformers import AutoModelForCausalLM as VLMClass

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
OUTPUT_DIR = Path("/kaggle/working/results/qwen3_mmhal")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"[{datetime.now().strftime('%H:%M:%S')}] Đang nạp mô hình {MODEL_ID}...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = VLMClass.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)
model.eval()
print("-> Đã nạp thành công mô hình lên GPU với định dạng bfloat16!")

## 4. Thực thi Inference trên toàn bộ 96 mẫu (Bảo đảm KHÔNG rò rỉ Ground-truth)

In [ ]:
responses = []
errors = []
log_lines = []

def log_msg(msg):
    print(msg)
    log_lines.append(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}")

log_msg(f"Bắt đầu thực thi suy luận trên {total_samples} mẫu...")
start_time_all = time.time()

for idx, sample in enumerate(dataset):
    sample_id = sample.get("image_id", idx)
    question = sample.get("question", "")
    gt_answer = sample.get("gt_answer", "")
    q_type = sample.get("question_type", "general")
    
    # Đọc ảnh từ thư mục images đã giải nén
    img_filename = sample["image_src"].split("/")[-1]
    img_path = DATA_DIR / "images" / img_filename
    if img_path.exists():
        image = Image.open(img_path).convert("RGB")
    else:
        image = Image.open(urllib.request.urlopen(sample["image_src"])).convert("RGB")

    t0 = time.time()
    try:
        # Chỉ truyền Ảnh + Câu hỏi vào mô hình (Loại bỏ triệt để ground truth)
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": question}
                ]
            }
        ]
        
        prompt_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[prompt_text], images=[image], padding=True, return_tensors="pt").to("cuda")

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False  # Greedy decoding chuẩn xác
            )

        trimmed_ids = [out[len(inp):] for inp, out in zip(inputs.input_ids, output_ids)]
        model_answer = processor.batch_decode(trimmed_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()
        elapsed_ms = int((time.time() - t0) * 1000)

        record = {
            "id": sample_id,
            "image": f"sample_{sample_id}",
            "question": question,
            "ground_truth": gt_answer,
            "model_answer": model_answer,
            "question_type": q_type,
            "inference_status": "SUCCESS"
        }
        responses.append(record)
        log_msg(f"[{idx+1}/{total_samples}] ID {sample_id} ({elapsed_ms}ms) -> {model_answer[:70]}...")

    except Exception as e:
        elapsed_ms = int((time.time() - t0) * 1000)
        log_msg(f"[{idx+1}/{total_samples}] ID {sample_id} ERROR: {str(e)}")
        record = {
            "id": sample_id,
            "image": f"sample_{sample_id}",
            "question": question,
            "ground_truth": gt_answer,
            "model_answer": "",
            "question_type": q_type,
            "inference_status": "FAILED"
        }
        responses.append(record)
        errors.append({"id": sample_id, "error": str(e)})

total_time = time.time() - start_time_all
log_msg(f"Hoàn thành suy luận {len(responses)} mẫu trong {total_time:.2f} giây.")

## 5. Xuất đầy đủ kết quả + mã nguồn (qwen3_vl.py, run_qwen3_mmhal.py) & Đóng gói ZIP

In [ ]:
import shutil

# 1. File qwen3_mmhal_responses.json
resp_path = OUTPUT_DIR / "qwen3_mmhal_responses.json"
with open(resp_path, "w", encoding="utf-8") as f:
    json.dump(responses, f, indent=2, ensure_ascii=False)

# 2. File inference_config.json
config_data = {
    "model_id": MODEL_ID,
    "dataset": "Shengcao1006/MMHal-Bench",
    "total_samples": total_samples,
    "target_dtype": "bfloat16",
    "generation_config": {
        "max_new_tokens": 512,
        "do_sample": False,
        "temperature": None,
        "top_p": None,
        "batch_size": 1
    },
    "timestamp": datetime.now(timezone.utc).isoformat()
}
with open(OUTPUT_DIR / "inference_config.json", "w", encoding="utf-8") as f:
    json.dump(config_data, f, indent=2, ensure_ascii=False)

# 3. File inference_summary.json
summary_data = {
    "total_samples": total_samples,
    "success_count": sum(1 for r in responses if r["inference_status"] == "SUCCESS"),
    "failed_count": sum(1 for r in responses if r["inference_status"] != "SUCCESS"),
    "duration_seconds": round(total_time, 2),
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
}
with open(OUTPUT_DIR / "inference_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_data, f, indent=2, ensure_ascii=False)

# 4. File inference_log.txt
with open(OUTPUT_DIR / "inference_log.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(log_lines) + "\n")

# 5. Tạo file mã nguồn qwen3_vl.py trong gói bàn giao
qwen3_vl_code = '''\"\"\"Qwen3-VL Adapter for MMHal-Bench.\"\"\"
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForVision2Seq

class Qwen3VLModel:
    def __init__(self, model_id="Qwen/Qwen3-VL-4B-Instruct", device=None, dtype=torch.bfloat16):
        self.model_id = model_id
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.dtype = dtype
        self.processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
        self.model = AutoModelForVision2Seq.from_pretrained(
            model_id, torch_dtype=self.dtype, device_map="auto" if self.device == "cuda" else None, trust_remote_code=True
        )
        self.model.eval()

    def infer(self, image: Image.Image, question: str, max_new_tokens: int = 512) -> str:
        messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": question}]}]
        prompt = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.processor(text=[prompt], images=[image], padding=True, return_tensors="pt").to(self.device)
        with torch.no_grad():
            output_ids = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        trimmed_ids = [out[len(inp):] for inp, out in zip(inputs.input_ids, output_ids)]
        return self.processor.batch_decode(trimmed_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()
'''
with open(OUTPUT_DIR / "qwen3_vl.py", "w", encoding="utf-8") as f:
    f.write(qwen3_vl_code)

# 6. Tạo file mã nguồn run_qwen3_mmhal.py trong gói bàn giao
run_code = '''\"\"\"Entry point script for Qwen3-VL on MMHal-Bench.\"\"\"
import json
from pathlib import Path
from qwen3_vl import Qwen3VLModel
from datasets import load_dataset

def main():
    print("Loading dataset MMHal-Bench...")
    dataset = load_dataset("Shengcao1006/MMHal-Bench", split="test")
    model = Qwen3VLModel()
    results = []
    for idx, sample in enumerate(dataset):
        s_id = sample.get("image_id", idx)
        q = sample.get("question", "")
        img = sample.get("image").convert("RGB")
        ans = model.infer(img, q)
        results.append({"id": s_id, "question": q, "ground_truth": sample.get("gt_answer", ""), "model_answer": ans, "inference_status": "SUCCESS"})
    with open("qwen3_mmhal_responses.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print("Inference completed.")

if __name__ == "__main__":
    main()
'''
with open(OUTPUT_DIR / "run_qwen3_mmhal.py", "w", encoding="utf-8") as f:
    f.write(run_code)

# 7. Tạo file README.md mô tả gói bàn giao
readme_code = '''# Qwen3-VL MMHal-Bench Deliverable Package\n\nNhiệm vụ: Chạy mô hình Qwen3-VL-4B-Instruct trên dataset MMHal-Bench.\n\n## Danh sách tệp:\n- `qwen3_mmhal_responses.json`: Kết quả suy luận của 96 mẫu MMHal-Bench.\n- `inference_config.json`: Cấu hình thực nghiệm chi tiết.\n- `inference_summary.json`: Báo cáo thống kê số mẫu thành công/thất bại.\n- `inference_log.txt`: Nhật ký tiến trình suy luận.\n- `qwen3_vl.py`: Model adapter cho Qwen3-VL.\n- `run_qwen3_mmhal.py`: Script thực thi suy luận.\n'''
with open(OUTPUT_DIR / "README.md", "w", encoding="utf-8") as f:
    f.write(readme_code)

# Đóng gói thành file ZIP tại thư mục gốc của Kaggle Output
zip_output = "/kaggle/working/qwen3_mmhal_results"
shutil.make_archive(zip_output, 'zip', OUTPUT_DIR)

print("\n=================== KẾT QUẢ THỰC HIỆN ===================")
print(f"File ZIP tải về: {zip_output}.zip")
print(f"Tổng mẫu: {summary_data['total_samples']} | Thành công: {summary_data['success_count']} | Thất bại: {summary_data['failed_count']}")
print("Các file có sẵn trong gói ZIP giải nén:")
print("  1. qwen3_mmhal_responses.json (Kết quả inference chính)")
print("  2. inference_config.json")
print("  3. inference_summary.json")
print("  4. inference_log.txt")
print("  5. qwen3_vl.py (Code model adapter)")
print("  6. run_qwen3_mmhal.py (Code runner)")
print("  7. README.md")
print("===========================================================")